In [1]:

# LazySummarizer Pro — Clean Setup

%pip install -q google-genai gradio

import importlib.metadata as metadata

print("✅ Dependencies ready")
print("google-genai:", metadata.version("google-genai"))
print("gradio:", metadata.version("gradio"))


✅ Dependencies ready
google-genai: 2.12.1
gradio: 6.30.0


In [2]:

# LazySummarizer Pro —  Test Gemma models

import time
from google import genai
from google.colab import userdata

try:
    API_KEY = userdata.get("GEMMA_API_KEY")
except Exception:
    API_KEY = None

if not API_KEY:
    raise ValueError(
        "GEMMA_API_KEY not found. Open Colab Secrets (🔑), "
        "add GEMMA_API_KEY, and enable Notebook access."
    )

client = genai.Client(api_key=API_KEY)

# Try both currently documented Gemma API models.
models_to_test = [
    "gemma-4-31b-it",
    "gemma-4-26b-a4b-it",
]

MODEL = None
last_error = None

for model_name in models_to_test:
    print(f"\nTesting {model_name}...")

    for attempt in range(1, 3):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=(
                    "Summarize in one sentence: "
                    "Rahul will draft slides by Friday at 5 PM."
                ),
                config={"temperature": 0.1}
            )

            if response.text:
                MODEL = model_name
                print("✅ Model working:", MODEL)
                print("Test response:", response.text)
                break

            print("Empty response; retrying...")

        except Exception as exc:
            last_error = exc
            print(
                f"Attempt {attempt} failed: "
                f"{type(exc).__name__}: {str(exc)[:250]}"
            )
            if attempt < 2:
                time.sleep(2 * attempt)

    if MODEL:
        break

if MODEL is None:
    print("\n❌ Both model tests failed.")
    print("Last error:", repr(last_error))
    print("Do not share your API key. Retry later if the error is 500/503.")
else:
    print("\n✅ LazySummarizer Pro can use this model.")



Testing gemma-4-31b-it...
Attempt 1 failed: ServerError: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}
Attempt 2 failed: ServerError: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}

Testing gemma-4-26b-a4b-it...
✅ Model working: gemma-4-26b-a4b-it
Test response: Rahul's slide draft is due Friday at 5 PM.

✅ LazySummarizer Pro can use this model.


In [3]:

# LazySummarizer Pro — Cell 3: Core Summarization

def generate_summary(chat_text: str) -> str:
    """Generate a structured summary from a group-chat transcript."""

    if not isinstance(chat_text, str) or not chat_text.strip():
        return "Please paste a chat transcript first."

    if len(chat_text) > 20000:
        return (
            "Transcript too long for this prototype. "
            "Please split it into smaller sections."
        )

    prompt = f"""
You are LazySummarizer Pro, an assistant that organizes long group chats.

ACCURACY RULES:
- Use only information explicitly stated in the transcript.
- Never invent names, dates, decisions, deadlines, or action items.
- A proposal is not a confirmed decision just because one person agrees.
- If confirmation is unclear, label the plan "Tentative".
- If an owner or deadline is missing, write "Not specified".
- Do not infer agreement from silence.
- Keep the result concise and easy to scan.

Use exactly these sections:

## 📝 Quick Summary
Give 3–5 concise bullet points.

## ✅ Confirmed Decisions
Include only clearly finalized decisions.
If none exist, write "None explicitly confirmed."

## 📌 Action Items
For each item include Task, Owner, and Deadline.
Use "Not specified" when information is missing.

## ❓ Open Questions
List unresolved questions and tentative plans requiring confirmation.

## 📅 Dates and Deadlines
Include only dates or times mentioned in the transcript.
Label tentative plans clearly.

GROUP CHAT TRANSCRIPT:
--- BEGIN TRANSCRIPT ---
{chat_text}
--- END TRANSCRIPT ---
"""

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config={"temperature": 0.1}
        )

        if not response.text:
            return "Gemma returned an empty response. Please try again."

        return response.text.strip()

    except Exception as exc:
        return (
            f"Gemma request failed: {type(exc).__name__}: {exc}\n\n"
            "Please try again in a moment."
        )


# Test with synthetic conversation data
sample_chat = """
Aanya: We need slides for the project presentation.
Rahul: I'll draft the slides by Friday at 5 PM.
Maya: I'll review them after Rahul uploads the draft.
Dev: Can we meet on Monday at 10 AM to rehearse?
Aanya: That works for me.
Maya: The venue is still not confirmed. Someone needs to check it.
Rahul: I'll share the final slide link when it's ready.
"""

print("⏳ Testing LazySummarizer Pro with", MODEL)
test_summary = generate_summary(sample_chat)
print("\n" + test_summary)


⏳ Testing LazySummarizer Pro with gemma-4-26b-a4b-it

## 📝 Quick Summary
* The team is preparing slides for a project presentation.
* Rahul is assigned to draft the slides.
* A rehearsal meeting is proposed for Monday at 10 AM.
* The venue for the rehearsal is currently unconfirmed.

## ✅ Confirmed Decisions
None explicitly confirmed.

## 📌 Action Items
* **Task:** Draft slides | **Owner:** Rahul | **Deadline:** Friday at 5 PM
* **Task:** Review slides | **Owner:** Maya | **Deadline:** Not specified
* **Task:** Check venue | **Owner:** Not specified | **Deadline:** Not specified
* **Task:** Share final slide link | **Owner:** Rahul | **Deadline:** Not specified

## ❓ Open Questions
* Will the rehearsal meeting on Monday at 10 AM be confirmed? (Tentative)
* What is the venue for the rehearsal?

## 📅 Dates and Deadlines
* Friday at 5 PM
* Monday at 10 AM (Tentative rehearsal meeting)


In [ ]:

# LazySummarizer Pro — Cell 4: Gradio Web App

import gradio as gr

with gr.Blocks(
    title="LazySummarizer Pro",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # 🦥 LazySummarizer Pro
    ### Your group chat, summarized in seconds.

    Turn long conversations into concise summaries, decisions,
    action items, deadlines, and unresolved questions.

    **Less scrolling. More doing.**
    """)

    with gr.Row():
        with gr.Column(scale=1):
            chat_input = gr.Textbox(
                label="💬 Group Chat Transcript",
                placeholder="Paste your group chat messages here...",
                lines=18,
                max_lines=30,
                value=sample_chat
            )

            with gr.Row():
                sample_btn = gr.Button("📝 Load Sample")
                clear_btn = gr.Button("🗑️ Clear")

            summarize_btn = gr.Button(
                "✨ Summarize Chat",
                variant="primary"
            )

        with gr.Column(scale=1):
            summary_output = gr.Markdown(
                value=(
                    "### 📋 Your Summary\n\n"
                    "Click **Summarize Chat** to generate your brief."
                )
            )

    summarize_btn.click(
        fn=generate_summary,
        inputs=[chat_input],
        outputs=[summary_output]
    )

    sample_btn.click(
        fn=lambda: sample_chat,
        inputs=[],
        outputs=[chat_input]
    )

    clear_btn.click(
        fn=lambda: (
            "",
            "Paste a conversation and click **Summarize Chat**."
        ),
        inputs=[],
        outputs=[chat_input, summary_output]
    )

    gr.Markdown("""
    ---
    **Privacy:** This prototype sends text to a hosted Gemma API.
    Do not paste private or sensitive messages without permission.

    **Limitations:** AI may omit or misinterpret details.
    Verify important decisions, names, and deadlines.
    """)

demo.launch(share=True, debug=True)


/tmp/ipykernel_753/869436843.py:5: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2db75db505c777b951.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
